In [1]:
CONFIG = {'project_root':'/home/featurize/work/biological_prior_validation', 'seed':20261008, 'score_threshold':700}


In [2]:
from pathlib import Path
import gzip, hashlib, json, shutil, sys, platform, subprocess
from urllib.request import Request, urlopen
from xml.sax.saxutils import escape
import numpy as np
import pandas as pd

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def save_json(p,obj):
    p=Path(p);temp=p.with_name(p.name+'.tmp')
    temp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str));temp.replace(p)

def validate_gzip(p):
    with gzip.open(p,'rb') as f:
        for _ in iter(lambda:f.read(1048576),b''):pass

def fetch_string(root,kind):
    name=f'9606.{kind}.v12.0.txt.gz'
    p=root/'data/raw/ppi'/name;p.parent.mkdir(parents=True,exist_ok=True)
    receipt=p.with_name(p.name+'.receipt.json')
    url=f'https://stringdb-downloads.org/download/{kind}.v12.0/{name}'
    if p.exists():
        if not receipt.exists():raise ValueError(f'Existing file has no receipt: {p}; preserve it before retrying')
        r=json.loads(receipt.read_text())
        if r['url']!=url or r['sha256']!=sha256(p):raise ValueError('STRING cache source/checksum mismatch')
        validate_gzip(p);return p
    tmp=p.with_name(p.name+'.part');n=0
    req=Request(url,headers={'User-Agent':'biological-prior-validation-step1'})
    with urlopen(req,timeout=90) as response,tmp.open('wb') as f:
        expected=response.headers.get('Content-Length')
        for b in iter(lambda:response.read(1048576),b''):
            n+=len(b)
            if n>128*1048576:raise ValueError('Unexpectedly large species-specific STRING file')
            f.write(b)
        if expected and n!=int(expected):raise ValueError('Incomplete STRING response')
    validate_gzip(tmp);digest=sha256(tmp);tmp.replace(p)
    save_json(receipt,dict(url=url,sha256=digest,size_bytes=n,gzip='full_CRC_checked'))
    return p

def map_ppi(genes,info,links,threshold=700):
    required={'protein1','protein2','combined_score'}
    if not required<=set(links):raise ValueError('STRING links schema mismatch')
    ids=[c for c in info if 'string_protein_id' in c.lower()]
    if len(ids)!=1 or 'preferred_name' not in info:raise ValueError('STRING info schema mismatch')
    if info[ids[0]].isna().any() or info[ids[0]].duplicated().any():raise ValueError('Ambiguous STRING protein identifiers')
    scores=pd.to_numeric(links.combined_score,errors='raise')
    if scores.isna().any() or not scores.between(0,1000).all() or not scores.eq(np.floor(scores)).all():raise ValueError('Invalid integer STRING score')
    names=info.preferred_name.astype('string').str.strip().replace('',pd.NA)
    mapping=dict(zip(info[ids[0]],names))
    universe=set(genes.gene_symbol.dropna())
    x=links.loc[scores.ge(threshold),['protein1','protein2']].copy();x['combined_score']=scores.loc[x.index].astype('int64')
    x['a']=x.protein1.map(mapping);x['b']=x.protein2.map(mapping)
    valid=x.a.notna() & x.b.notna();unmapped=int((~valid).sum());x=x.loc[valid]
    self_links=int(x.a.eq(x.b).sum());x=x.loc[x.a.ne(x.b) & x.a.isin(universe) & x.b.isin(universe)].copy()
    x['gene_a']=np.where(x.a.le(x.b),x.a,x.b);x['gene_b']=np.where(x.a.le(x.b),x.b,x.a)
    edges=x.groupby(['gene_a','gene_b'],as_index=False).combined_score.max().sort_values(['gene_a','gene_b']).reset_index(drop=True)
    edges['ppi_weight']=edges.combined_score/1000
    assert not edges.duplicated(['gene_a','gene_b']).any() and not edges.gene_a.eq(edges.gene_b).any()
    supported=set(edges.gene_a)|set(edges.gene_b)
    gm=genes.copy();gm['preferred_name_match']=gm.gene_symbol.isin(set(names.dropna()));gm['ppi_edge_supported']=gm.gene_symbol.isin(supported)
    gm['duplicated_expression_symbol']=gm.gene_symbol.notna() & gm.gene_symbol.duplicated(keep=False)
    summary=dict(status='PASS',STRING_version='12.0',species=9606,network='physical',score_threshold=threshold,
                 mapping='exact preferred_name match; no alias expansion',pair_rule='undirected canonical pair; maximum score',
                 population_scope='all_raw_annotation_symbols_reference_audit_not_selected_model_genes',
                 annotation_rows=len(genes),unique_annotation_symbols=len(universe),mapped_annotation_rows=int(gm.preferred_name_match.sum()),
                 supported_unique_genes=len(supported),undirected_edges=len(edges),unmapped_high_score_rows=unmapped,
                 self_symbol_rows_removed=self_links,model_gene_selection='NOT_RUN',model_graph='NOT_BUILT')
    return gm,edges,summary

def report_pdf(path, scan, alignment, ppi, totals):
    from xml.sax.saxutils import escape
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.cidfonts import UnicodeCIDFont
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.lib.pagesizes import A4
    from reportlab.lib import colors
    from reportlab.platypus import (SimpleDocTemplate, Paragraph, Table, TableStyle, PageBreak)
    pdfmetrics.registerFont(UnicodeCIDFont("STSong-Light"))
    style = ParagraphStyle("body",fontName="STSong-Light",fontSize=10,leading=16,wordWrap="CJK",spaceAfter=8,)
    title = ParagraphStyle("title", parent=style, fontSize=17, leading=23, spaceAfter=14)
    story = []
    def p(text):
        story.append(Paragraph(escape(str(text)), style))
    def table(rows):
        t = Table(
            [[Paragraph(escape(str(value)), style) for value in row]for row in rows],
            colWidths=[112, 112, 235],)
        t.setStyle(
            TableStyle([
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#e9f0f3")),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
                ("LINEBELOW", (0, 0), (-1, 0), 0.5, colors.HexColor("#305064")),
            ])
        )
        story.append(t)

    story.append(Paragraph("Step 1 Data Audit — Interim Report", title))
    p(
        "Dataset: GSM7056149 sciPlexGxE_2. Statistics come from the project's "
        "complete scan and cell-by-cell alignment; the large UMI file was not rescanned."
    )
    p(
        f"Original annotation: {scan['gene_annotation_rows']:,} gene rows and "
        f"{scan['candidate_barcode_rows']:,} candidate barcodes. The number of "
        "candidate barcodes is not the number of retained cells."
    )
    p(
        f"The complete scan found {scan['coordinate_rows']:,} count records and "
        f"{scan['total_umi']:,} total UMIs. Duplicate coordinates were summed; "
        "the record count does not represent the number of unique nonzero entries. "
        "The gzip CRC was checked, and the original SHA was compared with the "
        "download record."
    )
    p(
        f"At UMI ≥ {scan['min_umi']}, {alignment['raw_retained']:,} cells were "
        f"retained: {alignment['matched']:,} matched the authors' metadata and "
        f"{alignment['unmatched']:,} did not. Author-annotated cells outside the "
        f"retained set: {alignment['author_outside_raw_retained']}."
    )

    rows = [
        ["Cell line", "Treatment", "Cells in authors' metadata"]
    ] + totals[["cell_line", "drug", "n_cells"]].astype(str).values.tolist()
    table(rows)

    p(
        "Dose, sgRNA, target, replicate, and batch subdivisions are retained "
        "for every condition; see context_counts.csv. Unmatched cells are kept "
        "separately and are not assigned to a condition automatically."
    )

    story.append(PageBreak())
    story.append(Paragraph("Source Evidence, PPI, and Information Flow", title))

    p(
        "Replicate status: UNRESOLVED. Public code records the RT 1–192 / "
        "193–384 label partition and hash-collision filtering. Agreement between "
        "labels does not establish independent culture, transduction, or sampling. "
        "It also does not, by itself, establish technical replication."
    )
    p(
        "A hash plate number alone cannot serve as a global sample ID. Retain "
        "replicate, hash plate, well, and source together. PCR A–D are library "
        "index labels. Evidence connecting them to culture units is still missing."
    )
    p(
        f"PPI: {ppi.get('status', 'PENDING')}. Uses fixed human STRING v12.0 "
        "physical links with combined_score ≥ 700. Symbols are matched exactly "
        "to protein.info preferred_name; self-loops are removed; each undirected "
        "pair keeps its maximum score; edge weight is score/1000."
    )
    if ppi.get("status") == "PASS":
        p(
            "This check covers all original annotation symbols only: "
            f"{ppi['supported_unique_genes']:,} symbols with retained edges and "
            f"{ppi['undirected_edges']:,} undirected edges. It does not reproduce "
            "the old model's network for its 2,500 selected genes."
        )
    else:
        p(
            "The STRING download or mapping is incomplete. Errors and status "
            "were saved; this check must not be marked as passed."
        )

    p(
        "Old uploaded code: PPI-supported genes were selected using global "
        "logCP10K variance and detection rate. Global SVD, the cell graph, and "
        "metacell assignment occurred before the cell split. Later fitting on "
        "development cells alone cannot undo the exposure of upstream selection "
        "to evaluation-cell expression."
    )
    p(
        "This is a review of the old code's information flow, not a rerun or "
        "recalculation of its outputs. The new workflow has not yet performed "
        "normalization, HVG selection, metacell assignment, cell-graph construction, "
        "or count splitting."
    )
    p(
        "Next, review replicate_evidence.csv, preprocessing_audit.csv, "
        "ppi_audit_summary.json, and this report. Until biological independence "
        "is confirmed, do not describe the two directions as validation of "
        "generalization across independent biological replicates."
    )
    p(
        "Sources: GEO GSM7056149; the pinned commit and source-file SHA for "
        "cole-trapnell-lab/sci-Plex-GxE are in replicate_evidence.csv. STRING "
        "download URLs and SHAs are in receipts / step1_file_manifest.csv."
    )

    def footer(canvas, doc):
        canvas.setFont("STSong-Light", 9)
        canvas.drawString(45, 25, "biological_prior_validation | Step 1")
        canvas.drawRightString(A4[0] - 45, 25, str(doc.page))

    SimpleDocTemplate(
        str(path),
        pagesize=A4,
        rightMargin=45,
        leftMargin=45,
        topMargin=40,
        bottomMargin=40,
    ).build(story, onFirstPage=footer, onLaterPages=footer)

def finish_step1(cfg,evidence,legacy):
    if cfg['score_threshold']!=700:raise ValueError('This audit reproduces the documented threshold 700')
    root=Path(cfg['project_root']);audit=root/'data/metadata_audit';out=root/'results/raw'
    for p in [audit,out,root/'reports',root/'configs']:p.mkdir(parents=True,exist_ok=True)
    scan=json.loads((out/'fresh_count_scan_summary.json').read_text());a=json.loads((out/'fresh_alignment_summary.json').read_text())
    manifest=json.loads((audit/'download_manifest.json').read_text())
    assert scan['status']=='full_scan_complete' and scan['gzip_validation']=='full_stream_CRC_checked'
    assert scan['raw_sha256']==manifest['files']['GSM7056149_sciPlexGxE_2_UMI.count.matrix.gz']['sha256']
    assert scan['retained_cells']==a['raw_retained']==a['matched']+a['unmatched']
    assert a['author_metadata']==a['matched']+a['author_outside_raw_retained']
    totals=pd.read_csv(audit/'author_context_totals.csv');dose=pd.read_csv(audit/'author_dose_replicate_counts.csv');batch=pd.read_csv(audit/'replicate_batch_counts.csv',keep_default_na=False)
    assert int(totals.n_cells.sum())==int(dose.n_cells.sum())==a['author_metadata']
    assert int(batch.n_cells.sum())==a['raw_retained']
    pd.testing.assert_series_equal(dose.groupby(['cell_line','drug']).n_cells.sum().sort_index(),totals.set_index(['cell_line','drug']).n_cells.sort_index())
    assert dose.groupby(['cell_line','drug']).replicate.nunique().eq(2).all()
    evidence_df=pd.DataFrame(evidence);evidence_df.to_csv(audit/'replicate_evidence.csv',index=False)
    pd.DataFrame(legacy).to_csv(audit/'preprocessing_audit.csv',index=False)
    template=audit/'replicate_provenance_review.csv'
    if not template.exists():pd.DataFrame([dict(replicate=r,culture_unit='',transduction_unit='',culture_date='',source_file='',evidence_text='',review_status='MISSING') for r in ['replicate_1','replicate_2']]).to_csv(template,index=False)
    known=batch.loc[batch.replicate.ne('')]
    cross=known.groupby(['hash_plate','replicate','source_object'],as_index=False).n_cells.sum()
    cross.to_csv(audit/'hash_plate_replicate_crosswalk.csv',index=False)
    files=[]
    try:
        info_path=fetch_string(root,'protein.info');links_path=fetch_string(root,'protein.physical.links');files=[info_path,links_path]
        ann=root/'data/raw/GSM7056149_sciPlexGxE_2_gene.annotations.txt.gz'
        assert sha256(ann)==manifest['files'][ann.name]['sha256']
        genes=pd.read_csv(ann,sep='\t',header=None,dtype='string',keep_default_na=False)
        if genes.shape[1]!=2:raise ValueError('Expected two-column gene annotation')
        genes.columns=['gene_id','gene_symbol'];genes['gene_symbol']=genes.gene_symbol.str.strip().replace('',pd.NA);genes.insert(0,'gene_idx',np.arange(1,len(genes)+1))
        assert len(genes)==scan['gene_annotation_rows']
        gm,edges,ppi=map_ppi(genes,pd.read_csv(info_path,sep='\t'),pd.read_csv(links_path,sep=r'\s+'),cfg['score_threshold'])
        gm.to_csv(audit/'ppi_gene_mapping.csv',index=False);edges.to_parquet(root/'data/processed/ppi_annotation_reference_edges.parquet',index=False);files.append(ann)
    except Exception as e:
        ppi={'status':'ERROR','error':repr(e),'model_gene_selection':'NOT_RUN'}
        print('PPI audit incomplete:',repr(e))
    save_json(out/'ppi_audit_summary.json',ppi)
    rows=[dict(field=k,value=v,status='source_output_checked',source_file='fresh_count_scan_summary.json',scope='full_raw_count_scan') for k,v in scan.items()]
    rows += [dict(field=k,value=v,status='UNRESOLVED' if k=='biological_independence' else 'source_output_checked',source_file='fresh_alignment_summary.json',scope='raw_retained_alignment') for k,v in a.items() if k!='STRING']
    rows += [dict(field='ppi_'+k,value=v,status=ppi['status'],source_file='ppi_audit_summary.json',scope='annotation_reference_only') for k,v in ppi.items()]
    rows += [dict(field='legacy_'+r['stage'],value=r['rule'],status='source_code_review_not_rerun',source_file=r['source_notebook'],scope='prior_uploaded_pipeline') for r in legacy]
    rows += [dict(field='fine_condition_counts',value='dose/sgRNA/target/control/replicate/plate/well/library retained',status='derived_from_author_metadata',source_file='context_counts.csv',scope='raw_retained_matched_cells'),dict(field='identifiers',value='gene_idx and cell_idx are 1-based annotation row positions; barcodes matched one-to-one',status='source_code_and_outputs_checked',source_file='fresh_retained_cell_annotations.parquet',scope='raw_retained_mapping')]
    df=pd.DataFrame(rows);dest=audit/'metadata_audit.csv'
    if dest.exists():shutil.copyfile(dest,audit/('metadata_audit_before_04_'+sha256(dest)[:12]+'.csv'))
    df.to_csv(dest,index=False)
    if not (audit/'context_counts.csv').is_file():raise FileNotFoundError('Missing fine-grained context_counts.csv from notebook 03')
    files += [audit/'context_counts.csv',out/'fresh_count_scan_summary.json',out/'fresh_alignment_summary.json',audit/'replicate_evidence.csv',audit/'preprocessing_audit.csv',audit/'metadata_audit.csv']
    pd.DataFrame([dict(path=str(p),size_bytes=p.stat().st_size,sha256=sha256(p)) for p in files]).to_csv(audit/'step1_file_manifest.csv',index=False)
    save_json(root/'configs/step1_completion.json',cfg)
    try:commit=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],stderr=subprocess.DEVNULL,text=True).strip()
    except (OSError,subprocess.CalledProcessError):commit='not_available'
    save_json(out/'step1_completion_environment.json',dict(python=sys.version,platform=platform.platform(),numpy=np.__version__,pandas=pd.__version__,seed=cfg['seed'],git_commit=commit))
    report_pdf(root/'reports/data_audit.pdf',scan,a,ppi,totals)
    print('Saved:',root/'reports/data_audit.pdf','| PPI:',ppi['status'],'| biological independence: UNRESOLVED')
    return df,ppi


In [3]:
EVIDENCE = [{'source': 'public_author_code', 'commit': '63eed120697611d8baf4a207a797a147308a6674', 'url': 'https://github.com/cole-trapnell-lab/sci-Plex-GxE/blob/63eed120697611d8baf4a207a797a147308a6674/Kinome_GxE_screen/Notebook_1_cds_clenaup.R#L25', 'file': 'Notebook_1_cds_clenaup.R', 'line': 25, 'source_sha256': '870f0ecaafbdc3762e425426ef6dc6b76b680a6a1d8b14fc622d2118f7177703', 'evidence_text': 'colData(cds)$background_loading <- background_projections_df$background_projections', 'interpretation': 'Code/label or QC evidence; independent culture/transduction not established'}, {'source': 'public_author_code', 'commit': '63eed120697611d8baf4a207a797a147308a6674', 'url': 'https://github.com/cole-trapnell-lab/sci-Plex-GxE/blob/63eed120697611d8baf4a207a797a147308a6674/Kinome_GxE_screen/Notebook_1_cds_clenaup.R#L29', 'file': 'Notebook_1_cds_clenaup.R', 'line': 29, 'source_sha256': '870f0ecaafbdc3762e425426ef6dc6b76b680a6a1d8b14fc622d2118f7177703', 'evidence_text': '### Cleaneup hash data and remove any cells with hashes mapped from a different replicate', 'interpretation': 'Code/label or QC evidence; independent culture/transduction not established'}, {'source': 'public_author_code', 'commit': '63eed120697611d8baf4a207a797a147308a6674', 'url': 'https://github.com/cole-trapnell-lab/sci-Plex-GxE/blob/63eed120697611d8baf4a207a797a147308a6674/Kinome_GxE_screen/Notebook_1_cds_clenaup.R#L30', 'file': 'Notebook_1_cds_clenaup.R', 'line': 30, 'source_sha256': '870f0ecaafbdc3762e425426ef6dc6b76b680a6a1d8b14fc622d2118f7177703', 'evidence_text': '### i.e., remove rare instances where a cell in replicate 1, RT wells 1 to 192, has a', 'interpretation': 'Code/label or QC evidence; independent culture/transduction not established'}, {'source': 'public_author_code', 'commit': '63eed120697611d8baf4a207a797a147308a6674', 'url': 'https://github.com/cole-trapnell-lab/sci-Plex-GxE/blob/63eed120697611d8baf4a207a797a147308a6674/Kinome_GxE_screen/Notebook_1_cds_clenaup.R#L31', 'file': 'Notebook_1_cds_clenaup.R', 'line': 31, 'source_sha256': '870f0ecaafbdc3762e425426ef6dc6b76b680a6a1d8b14fc622d2118f7177703', 'evidence_text': '### hash assigned to replicate 2, RT wells 193 to 384, or vice-versa', 'interpretation': 'Code/label or QC evidence; independent culture/transduction not established'}, {'source': 'public_author_code', 'commit': '63eed120697611d8baf4a207a797a147308a6674', 'url': 'https://github.com/cole-trapnell-lab/sci-Plex-GxE/blob/63eed120697611d8baf4a207a797a147308a6674/Kinome_GxE_screen/Notebook_1_cds_clenaup.R#L37', 'file': 'Notebook_1_cds_clenaup.R', 'line': 37, 'source_sha256': '870f0ecaafbdc3762e425426ef6dc6b76b680a6a1d8b14fc622d2118f7177703', 'evidence_text': 'rep1_colData <- colData(cds)[colData(cds)$RT_well %in%  c(1:192),]', 'interpretation': 'Code/label or QC evidence; independent culture/transduction not established'}, {'source': 'public_author_code', 'commit': '63eed120697611d8baf4a207a797a147308a6674', 'url': 'https://github.com/cole-trapnell-lab/sci-Plex-GxE/blob/63eed120697611d8baf4a207a797a147308a6674/Kinome_GxE_screen/Notebook_1_cds_clenaup.R#L45', 'file': 'Notebook_1_cds_clenaup.R', 'line': 45, 'source_sha256': '870f0ecaafbdc3762e425426ef6dc6b76b680a6a1d8b14fc622d2118f7177703', 'evidence_text': 'rep2_colData <- colData(cds)[colData(cds)$RT_well %in%  c(193:384),]', 'interpretation': 'Code/label or QC evidence; independent culture/transduction not established'}, {'source': 'public_author_code', 'commit': '63eed120697611d8baf4a207a797a147308a6674', 'url': 'https://github.com/cole-trapnell-lab/sci-Plex-GxE/blob/63eed120697611d8baf4a207a797a147308a6674/Kinome_GxE_screen/Notebook_1_cds_clenaup.R#L60', 'file': 'Notebook_1_cds_clenaup.R', 'line': 60, 'source_sha256': '870f0ecaafbdc3762e425426ef6dc6b76b680a6a1d8b14fc622d2118f7177703', 'evidence_text': '# Remove hash replicate collisions -> 1051983 (99.99789%)', 'interpretation': 'Code/label or QC evidence; independent culture/transduction not established'}, {'source': 'public_author_code', 'commit': '63eed120697611d8baf4a207a797a147308a6674', 'url': 'https://github.com/cole-trapnell-lab/sci-Plex-GxE/blob/63eed120697611d8baf4a207a797a147308a6674/Kinome_GxE_screen/Notebook_2_Doublet_removal_and_hash_filtering.R#L26', 'file': 'Notebook_2_Doublet_removal_and_hash_filtering.R', 'line': 26, 'source_sha256': '175056d556a2a11eed32d1c0e0079d234ddb0ad29844744ca2e651fef7c5a58c', 'evidence_text': '### Filter doublets using scrublet', 'interpretation': 'Code/label or QC evidence; independent culture/transduction not established'}, {'source': 'public_author_code', 'commit': '63eed120697611d8baf4a207a797a147308a6674', 'url': 'https://github.com/cole-trapnell-lab/sci-Plex-GxE/blob/63eed120697611d8baf4a207a797a147308a6674/Kinome_GxE_screen/Notebook_2_Doublet_removal_and_hash_filtering.R#L48', 'file': 'Notebook_2_Doublet_removal_and_hash_filtering.R', 'line': 48, 'source_sha256': '175056d556a2a11eed32d1c0e0079d234ddb0ad29844744ca2e651fef7c5a58c', 'evidence_text': '### Number of cells after doublet removal = 1042721 (0.8804325% inferred doublet rate)', 'interpretation': 'Code/label or QC evidence; independent culture/transduction not established'}, {'source': 'public_author_code', 'commit': '63eed120697611d8baf4a207a797a147308a6674', 'url': 'https://github.com/cole-trapnell-lab/sci-Plex-GxE/blob/63eed120697611d8baf4a207a797a147308a6674/Kinome_GxE_screen/Notebook_2_Doublet_removal_and_hash_filtering.R#L67', 'file': 'Notebook_2_Doublet_removal_and_hash_filtering.R', 'line': 67, 'source_sha256': '175056d556a2a11eed32d1c0e0079d234ddb0ad29844744ca2e651fef7c5a58c', 'evidence_text': '### Filter doublets using hashes. Number of cells after hash filtering 991940 (4.870047% empirical doublet rate)', 'interpretation': 'Code/label or QC evidence; independent culture/transduction not established'}, {'source': 'audit_limitation', 'commit': '63eed120697611d8baf4a207a797a147308a6674', 'url': '', 'file': '', 'line': '', 'source_sha256': '', 'evidence_text': 'No verified replicate-to-independent-culture/transduction mapping supplied', 'interpretation': 'UNRESOLVED; neither independent biological nor technical status inferred'}]
LEGACY = [{'stage': 'cell_calling', 'rule': 'total raw UMI >=500; raw indices are one-based', 'source_notebook': '02_cell_calling.ipynb', 'evidence_level': 'prior_uploaded_source_code_review_not_execution'}, {'stage': 'barcode_mapping', 'rule': 'raw annotation row position = cell_idx; one-to-one retained join', 'source_notebook': '03_build_cell_metadata.ipynb', 'evidence_level': 'prior_uploaded_source_code_review_not_execution'}, {'stage': 'STRING_source', 'rule': 'v12.0 Homo sapiens physical links; combined_score >=700', 'source_notebook': '05_select_final_ppi_genes.ipynb', 'evidence_level': 'prior_uploaded_source_code_review_not_execution'}, {'stage': 'protein_gene_mapping', 'rule': 'protein.info preferred_name; remove unmapped and self links; canonical pair max score', 'source_notebook': '05_select_final_ppi_genes.ipynb', 'evidence_level': 'prior_uploaded_source_code_review_not_execution'}, {'stage': 'gene_selection', 'rule': 'PPI-supported expression genes ranked by logCP10K sample variance then detection_fraction; one gene_symbol; top 2500', 'source_notebook': '05_select_final_ppi_genes.ipynb', 'evidence_level': 'prior_uploaded_source_code_review_not_execution'}, {'stage': 'normalization', 'rule': 'log1p(10000*count/total raw UMI across all genes)', 'source_notebook': '06_build_final_expression_and_ppi_graph.ipynb', 'evidence_level': 'prior_uploaded_source_code_review_not_execution'}, {'stage': 'PPI_weights', 'rule': 'combined_score/1000; symmetric W; W_scaled=W/lambda_max(L_raw)', 'source_notebook': '05/06 notebooks', 'evidence_level': 'prior_uploaded_source_code_review_not_execution'}, {'stage': 'cell_graph', 'rule': 'global selected-expression SVD50, n_iter=2, seed=2026, cosine/HNSW k=30', 'source_notebook': '07_build_cell_graph.ipynb', 'evidence_level': 'prior_uploaded_source_code_review_not_execution'}, {'stage': 'metacell_split', 'rule': 'global MiniBatchKMeans metacells before within-metacell cell split', 'source_notebook': '08_fixed_dualgraph.ipynb', 'evidence_level': 'prior_uploaded_source_code_review_not_execution'}, {'stage': 'initial_information_flow', 'rule': 'gene selection, SVD/graph and metacell assignment see future test cells before split', 'source_notebook': '05/07/08 notebooks', 'evidence_level': 'prior_uploaded_source_code_review_not_execution'}, {'stage': 'cross_replicate_upstream', 'rule': 'later dev-only fitting still uses globally selected genes and earlier normalized matrix', 'source_notebook': '05/06 plus earlier 19/20/21 notebooks', 'evidence_level': 'prior_uploaded_source_code_review_not_execution'}, {'stage': 'invalid_legacy_header', 'rule': '01 treats coordinate records as MatrixMarket header/dimensions; do not reuse dimension CSV', 'source_notebook': '01_check_raw_data.ipynb', 'evidence_level': 'prior_uploaded_source_code_review_not_execution'}]


In [4]:
from IPython.display import display
metadata_audit, ppi_summary = finish_step1(CONFIG, EVIDENCE, LEGACY)
display(metadata_audit)
print(json.dumps(ppi_summary,ensure_ascii=False,indent=2))
display(pd.read_csv(Path(CONFIG['project_root'])/'data/metadata_audit/hash_plate_replicate_crosswalk.csv'))


Saved: /home/featurize/work/biological_prior_validation/reports/data_audit.pdf | PPI: PASS | biological independence: UNRESOLVED


,field,value,status,source_file,scope
0,status,full_scan_complete,source_output_checked,fresh_count_scan_summary.json,full_raw_count_scan
1,coordinate_rows,1879036606,source_output_checked,fresh_count_scan_summary.json,full_raw_count_scan
2,total_umi,3649095757,source_output_checked,fresh_count_scan_summary.json,full_raw_count_scan
3,raw_sha256,a1443d71dc890056bfb804d57bdef19d21942563ff088d...,source_output_checked,fresh_count_scan_summary.json,full_raw_count_scan
4,gene_annotation_rows,58347,source_output_checked,fresh_count_scan_summary.json,full_raw_count_scan
5,candidate_barcode_rows,43209765,source_output_checked,fresh_count_scan_summary.json,full_raw_count_scan
6,gzip_validation,full_stream_CRC_checked,source_output_checked,fresh_count_scan_summary.json,full_raw_count_scan
7,duplicate_policy,additive; coordinate rows are not unique nnz,source_output_checked,fresh_count_scan_summary.json,full_raw_count_scan
8,min_umi,500,source_output_checked,fresh_count_scan_summary.json,full_raw_count_scan
9,retained_cells,1052205,source_output_checked,fresh_count_scan_summary.json,full_raw_count_scan


{
  "status": "PASS",
  "STRING_version": "12.0",
  "species": 9606,
  "network": "physical",
  "score_threshold": 700,
  "mapping": "exact preferred_name match; no alias expansion",
  "pair_rule": "undirected canonical pair; maximum score",
  "population_scope": "all_raw_annotation_symbols_reference_audit_not_selected_model_genes",
  "annotation_rows": 58347,
  "unique_annotation_symbols": 56648,
  "mapped_annotation_rows": 18825,
  "supported_unique_genes": 10388,
  "undirected_edges": 77053,
  "unmapped_high_score_rows": 0,
  "self_symbol_rows_removed": 0,
  "model_gene_selection": "NOT_RUN",
  "model_graph": "NOT_BUILT"
}


,hash_plate,replicate,source_object,n_cells
0,10,replicate_1,A172,37796
1,10,replicate_2,T98G,29849
2,11,replicate_1,A172,37776
3,14,replicate_1,U87MG,36364
4,14,replicate_2,T98G,44326
5,15,replicate_1,U87MG,34620
6,15,replicate_2,A172,27592
7,16,replicate_1,U87MG,31297
8,16,replicate_2,T98G,24986
9,17,replicate_1,U87MG,28689
